In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/sample_submission.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json


In [2]:
import json
import os
import glob
from collections import deque

print("=== Starting ARC Fluid Intelligence Dual-Attempt Kaggle Submission Generator ===")

# ---------------------------------------------------------
# 1. Locate ARC Challenges Dataset File on Kaggle
# ---------------------------------------------------------
possible_paths = [
    "/kaggle/input/arc-prize-2024/arc-agi_evaluation_challenges.json",
    "/kaggle/input/arc-prize-2024/arc-agi_test_challenges.json",
    "/kaggle/input/abstraction-and-reasoning-corpus/arc-agi_evaluation_challenges.json",
    "/kaggle/input/abstraction-and-reasoning-corpus/arc-agi_test_challenges.json",
    "./arc-agi_evaluation_challenges.json",
    "./challenges.json"
]

challenge_file = None
for path in possible_paths:
    if os.path.exists(path):
        challenge_file = path
        break

if not challenge_file:
    found = glob.glob("/kaggle/input/**/*.json", recursive=True)
    for f in found:
        if "challenge" in f.lower() or "test" in f.lower() or "evaluation" in f.lower():
            challenge_file = f
            break

if challenge_file:
    print(f"Loading challenges from: {challenge_file}")
    with open(challenge_file, "r") as f:
        tasks_data = json.load(f)
else:
    print("Warning: Challenge file not found in input paths. Creating synthetic benchmark tasks...")
    tasks_data = {
        "00576224": {
            "train": [{"input": [[0, 3, 0], [3, 3, 3], [0, 3, 0]], "output": [[0, 3, 0], [3, 8, 3], [0, 3, 0]]}],
            "test": [{"input": [[0, 3, 0], [3, 3, 3], [0, 3, 0]]}]
        },
        "009d5c81": {
            "train": [{"input": [[1, 0, 1], [0, 1, 0], [1, 0, 1]], "output": [[1, 1, 1], [1, 0, 1], [1, 1, 1]]}],
            "test": [{"input": [[1, 0, 1], [0, 1, 0], [1, 0, 1]]}]
        },
        "12997ef3": {
            "train": [{"input": [[2, 0, 0], [0, 2, 0], [0, 0, 2]], "output": [[2, 2, 2], [2, 2, 2], [2, 2, 2]]}],
            "test": [
                {"input": [[2, 0, 0], [0, 2, 0], [0, 0, 2]]},
                {"input": [[0, 2, 0], [2, 0, 2], [0, 2, 0]]}
            ]
        }
    }

print(f"Total Tasks to Process: {len(tasks_data)}")

# ---------------------------------------------------------
# 2. Spatial & DSL Operators Engine
# ---------------------------------------------------------
def rotate_90(grid):
    R, C = len(grid), len(grid[0]) if len(grid) > 0 else 0
    res = [[0]*R for _ in range(C)]
    for r in range(R):
        for c in range(C):
            res[c][R - 1 - r] = grid[r][c]
    return res

def mirror_horizontal(grid):
    return [row[::-1] for row in grid]

def mirror_vertical(grid):
    return grid[::-1]

def apply_gravity(grid):
    R, C = len(grid), len(grid[0]) if len(grid) > 0 else 0
    res = [row[:] for row in grid]
    for c in range(C):
        col_vals = [res[r][c] for r in range(R) if res[r][c] != 0]
        num_zeros = R - len(col_vals)
        for r in range(num_zeros):
            res[r][c] = 0
        for r in range(len(col_vals)):
            res[num_zeros + r][c] = col_vals[r]
    return res

def flood_fill_holes(grid, fill_color=8):
    R, C = len(grid), len(grid[0]) if len(grid) > 0 else 0
    res = [row[:] for row in grid]
    visited = [[False]*C for _ in range(R)]
    q = deque()
    for r in range(R):
        for c in range(C):
            if (r == 0 or r == R - 1 or c == 0 or c == C - 1) and res[r][c] == 0:
                q.append((r, c))
                visited[r][c] = True
    while q:
        r, c = q.popleft()
        for nr, nc in [(r+1, c), (r-1, c), (r, c+1), (r, c-1)]:
            if 0 <= nr < R and 0 <= nc < C and not visited[nr][nc] and res[nr][nc] == 0:
                visited[nr][nc] = True
                q.append((nr, nc))
    for r in range(R):
        for c in range(C):
            if res[r][c] == 0 and not visited[r][c]:
                res[r][c] = fill_color
    return res

OPERATORS = [
    ("identity", lambda g: [r[:] for r in g]),
    ("gravity", apply_gravity),
    ("mirror_h", mirror_horizontal),
    ("mirror_v", mirror_vertical),
    ("rotate_90", rotate_90),
    ("rotate_180", lambda g: rotate_90(rotate_90(g))),
    ("flood_fill", flood_fill_holes)
]

def grids_equal(g1, g2):
    if len(g1) != len(g2) or (len(g1) > 0 and len(g1[0]) != len(g2[0])): return False
    for r in range(len(g1)):
        for c in range(len(g1[0])):
            if g1[r][c] != g2[r][c]: return False
    return True

def learn_best_operators(train_pairs):
    if not train_pairs: return [OPERATORS[0]]
    scored = []
    for name, op_func in OPERATORS:
        matches = 0
        for pair in train_pairs:
            inp, out = pair.get("input"), pair.get("output")
            if inp and out:
                try:
                    if grids_equal(op_func(inp), out): matches += 1
                except Exception: pass
        scored.append((matches, name, op_func))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored

def predict_dual_attempts(task_data, test_input_grid):
    train_pairs = task_data.get("train", [])
    ranked = learn_best_operators(train_pairs)
    best_op = ranked[0][2]
    
    try:
        attempt_1 = best_op(test_input_grid)
    except Exception:
        attempt_1 = [r[:] for r in test_input_grid]
    
    if len(ranked) > 1 and ranked[1][0] > 0:
        try:
            attempt_2 = ranked[1][2](test_input_grid)
        except Exception:
            attempt_2 = rotate_90(test_input_grid)
    else:
        attempt_2 = rotate_90(test_input_grid)
        
    if grids_equal(attempt_1, attempt_2):
        attempt_2 = mirror_horizontal(test_input_grid)
        
    return {"attempt_1": attempt_1, "attempt_2": attempt_2}

# ---------------------------------------------------------
# 3. Generate Submission Dictionary & Save File
# ---------------------------------------------------------
submission = {}

for task_id, task in tasks_data.items():
    submission[task_id] = []
    for test_item in task.get("test", []):
        inp_grid = test_item.get("input", [[0]])
        submission[task_id].append(predict_dual_attempts(task, inp_grid))

out_path = "/kaggle/working/submission.json" if os.path.exists("/kaggle/working") else "submission.json"
with open(out_path, "w") as f:
    json.dump(submission, f)

print(f"✅ Successfully exported submission.json with {len(submission)} tasks to: {out_path}")

# Verify structural correctness
sample_id = list(submission.keys())[0]
print(f"Sample task format validation for '{sample_id}':")
print(json.dumps({sample_id: submission[sample_id]}, indent=2))

=== Starting ARC Fluid Intelligence Dual-Attempt Kaggle Submission Generator ===
Loading challenges from: /kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json
Total Tasks to Process: 120


AttributeError: 'list' object has no attribute 'get'